# GP path planning (L3) on the shapes sample -- 2026-11-09

Working copy (`currently working on`). Same code path as `Run_SelfDriving_Scan.ipynb`, plus:
- a readback-length check (tip withdrawn) before the run,
- `executor.tail_dwell_s`: every waveform ends at rest for a moment so PyScanner's short AI record never cuts the path,
- the equivalent raster as **user-defined, drive-corrected** waveforms, cut only at line ends (`user_raster`).

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # sklearn GP kernel-bound convergence warnings are benign

import time
import pathlib
import numpy as np
import matplotlib
%matplotlib widget
import matplotlib.pyplot as plt

from selfdriving_afm.config import ExperimentConfig
from selfdriving_afm.execution import ContinuousExecutor
from selfdriving_afm.planners.edge_follow_planner import EdgeFollowPlanner
from selfdriving_afm.planners.gp_path_planner import GPPathPlanner
from selfdriving_afm.planners.edge_explorer_planner import EdgeExplorerPlanner
from selfdriving_afm.planners.grid_refine_planner import GridRefinePlanner
from selfdriving_afm.viz import (LiveEdgeFollowView, LiveGPPathView, LiveGridRefineView, build_grid,
                                  reconstruct_topography, subsample_for_reconstruction,
                                  bounds_from_history, grid_from_bounds)
from selfdriving_afm.reconstruction import reconstruct_image
from selfdriving_afm import modes  # the scan modes -- shared with Simulation.ipynb


In [ ]:
USE_HARDWARE = True  # <-- flip to True on the instrument

if USE_HARDWARE:
    import json
    import pathlib
    from selfdriving_afm.backends.pyscanner_backend import connect, PyScannerBackend

    HARDWARE_PATHS_FILE = pathlib.Path("hardware_paths.json")  # per-computer, gitignored
    if not HARDWARE_PATHS_FILE.exists():
        raise FileNotFoundError(
            f"{HARDWARE_PATHS_FILE.resolve()} not found. Copy hardware_paths.example.json "
            f"to hardware_paths.json (next to this notebook) and edit exe_path/vi_path "
            f"for THIS computer -- every microscope's PyScanner install lives somewhere different."
        )
    hw_paths = json.loads(HARDWARE_PATHS_FILE.read_text())

    vi = connect(exe_path=hw_paths["exe_path"], vi_path=hw_paths["vi_path"])
    # the Asylum Research software (Igor Pro), over COM: engage / withdraw, crosspoint,
    # X/Y loops (section 3a)
    from selfdriving_afm import asylum
    igor = asylum.connect_igor()
else:
    from selfdriving_afm.backends.simulate_backend import SimulateBackend, default_topography
    from selfdriving_afm import asylum
    igor = None   # no Igor: section 3a prints the commands without sending them


In [ ]:
cfg = ExperimentConfig()

# --- where this run's data gets saved -- created now if it doesn't exist yet, so a
# permissions problem shows up here rather than after a long scan finishes ---
OUTPUT_DIR = pathlib.Path("scan_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODALITY = "gp_path"  # "edge_follow", "gp_path", "edge_explorer" or "grid_refine"  <-- TOGGLE SELF-DRIVING MODE HERE

#
# --- tip velocity: the ONE knob that sets how fast the tip moves, everywhere
# (spirals, circles, GP path segments, raster/Lissajous/spiral scans incl. raster
# turnarounds, moves between scans, and the return to origin). It is the real speed
# only if sample_rate_hz = PyScanner's data_rate -- checked when the executor is built.
# Point density is derived from this + sample_rate_hz, not set separately -- and it
# scales as arc_length * sample_rate_hz / tip_velocity_V_s, so a slow tip on a fast
# DAQ over a large search area can need tens of millions of samples for one spiral
# (trajectory.py raises a clear error above 10,000,000 points/primitive rather than
# hanging -- if you hit that, raise tip_velocity_V_s, lower sample_rate_hz, or
# shrink max_search_radius_V/path_segment_length_V).
#
# The values below are chosen to run FAST IN SIMULATION against this project's small
# volt-scale synthetic sample -- they are NOT a real tip speed or DAQ rate. Before a
# real run, replace both with your actual instrument numbers (CONFIRM ON HARDWARE --
# see README "Tip velocity"); real hardware will likely want a much slower
# tip_velocity_V_s than this.
cfg.velocity.tip_velocity_V_s = 30
cfg.velocity.sample_rate_hz = 1e6 #1.0e6 or 0.1e6 or 0.01e6

# --- edge-follow knobs (only used if MODALITY == "edge_follow") ---
# search spirals: the FIRST one covers the whole area to explore, and the spiral turn
# spacing and the tracing-circle sizes come from the feature sizes -- all set by the
# priors (section 2b). Every edge a spiral crosses gets traced; then the next spiral goes
# to the nearest area not scanned yet (or grows -- see spiral_search_mode below)
cfg.edge_follow.spiral_growth_factor = 1.5        # how much bigger each retry spiral gets ("grow" mode only)
cfg.edge_follow.max_search_radius_V = 4.0         # give up growing the spiral past this ("grow" mode only)
# what a spiral that finds nothing new does next:
cfg.edge_follow.spiral_search_mode = "relocate"          # "relocate" = next spiral (same radius) goes to the
                                                          #   nearest area not scanned yet; "grow" = old behaviour
                                                          #   (bigger spiral, same center)
cfg.edge_follow.first_scan_pattern = "spiral"             # FIRST search: "spiral" (the circle inside the area), or
                                                          #   "raster" / "lissajous" over the whole square area, corners
                                                          #   included (lines 0.5 x smallest feature apart, from the
                                                          #   priors); later searches are always spirals
cfg.edge_follow.explore_min_unexplored_fraction = 0.5     # a new spiral spot must be at least this much unexplored
                                                          #   (lower = spirals overlap more but miss less)
cfg.edge_follow.spiral.n_cycles = 6             # how tightly wound the spiral is (the priors
                                                  #   replace this with turns 0.5 x smallest feature apart)
cfg.edge_follow.spiral.direction = "ccw"          # "ccw" or "cw"
# edge detection -- what counts as "found an edge" while spiraling/tracing
cfg.edge_follow.edge_detect.mode = "manual"           # "auto_mad" or "manual" + manual_threshold
cfg.edge_follow.edge_detect.manual_threshold = 100     # required if mode == "manual"
cfg.edge_follow.edge_detect.mad_k = 6.0                 # higher = less sensitive (fewer false edges)
# tracing circles, once the spiral finds an edge -- their radius comes from the priors
# (section 2b: sized from the measured edge curvature, between limits from the feature sizes)
cfg.edge_follow.circle.overlap_fraction = 0.6
# one edge = one feature: what happens when a tracing circle sees a SEPARATE edge
cfg.edge_follow.on_new_edge = "switch"          # "switch" = trace the new edge now, then come back
                                                  #   and finish this one; "queue" = finish this one first
cfg.edge_follow.trace_both_directions = True     # edge lost one way -> follow it the other way too (same feature)
cfg.edge_follow.same_feature_distance_V = None   # closer than this to a traced edge = same edge
                                                  #   (None = a quarter of the circle radius, measured from the traced edge line)
cfg.edge_follow.max_circles_per_feature = 150    # per direction along an edge
cfg.edge_follow.edge_detect.min_run_V = 0.002    # on/off stretches shorter than this are noise, not edges
#
# --- stopping criteria: several apply at once, whichever fires first wins. Toggle
# any of them off by setting it to None. See README "When does a scan stop?" for
# the full reasoning behind each one.
cfg.edge_follow.max_features = 5           # (1) stop once this many features are traced
                                             #     ("search-exhausted" -- no edge found even
                                             #     after growing to max_search_radius_V -- is
                                             #     always on too, it's how the planner gives up)
cfg.edge_follow.max_run_time_s = None      # (2) TIME: wall-clock cap in seconds, e.g. 120 = 2
                                             #     minutes. Set to None to disable and rely only
                                             #     on max_features / search exhaustion above.

# --- GP path knobs (only used if MODALITY == "gp_path") ---
# step length -- how far each committed path segment goes
cfg.gp_path.initial_path_length_V = 0.5    # length of the very FIRST segment (no GP fit
                                              # exists yet to pick a direction from, so this
                                              # one is just a fixed-length probe)
cfg.gp_path.path_segment_length_V = 0.5    # length of every segment AFTER the first --
                                              # THIS is "the GP step size" -- bigger covers
                                              # ground faster but coarser gradient estimates;
                                              # smaller is finer but needs more segments
# direction search -- how the planner picks where each segment goes
cfg.gp_path.n_candidate_directions = 16    # how many directions are scored at each step
                                              # (evenly spaced) before committing to the best
cfg.gp_path.exploration_weight = 1.5        # candidate score = edge term + this * exploration term
                                              # -- higher favors exploring unknown area over
                                              # following the steepest known gradient
cfg.gp_path.gradient_eval_resolution_V = 0.005  # internal only -- how finely candidate
                                              # directions are sampled against the fitted GP.
                                              # Never touches hardware; no need to change this
# GP back end -- THE speed knob: how much work each refit does (see README "GP back
# ends"). Benchmarked at these settings (~6000 points after 60 segments, 2-core CPU):
#   "sklearn_exact" = the original: re-learns the kernel on ALL points every refit --
#                     ~20 min per refit at 6000 points, the tip waits minutes per segment
#   "frozen"        = kernel learned until freeze_after_n points, then frozen -- seconds
#   "local"         = frozen kernel, exact GP on the local_n_points nearest the tip --
#                     ~0.05 s per segment at any run length
#   "sparse"        = frozen kernel, sparse GP over everywhere the path has been --
#                     ~0.03 s per segment, made the same choices as sklearn_exact
# "sparse"/"local" keep every segment well under a second on a plain CPU (no GPU)
cfg.gp_path.gp_backend = "sparse"           # "sklearn_exact", "frozen", "local" or "sparse"
cfg.gp_path.refit_every = 1                 # refit after every segment (cheap with sparse/local;
                                              # use 5 with "sklearn_exact" as before)
cfg.gp_path.freeze_after_n = 1500           # learn the kernel until this many points (each
                                              # learn ~0.4 s), then keep it fixed
cfg.gp_path.hyper_opt_subset = 600          # kernel learned on at most this many random points
cfg.gp_path.local_n_points = 1000           # "local" only: points per refit
cfg.gp_path.sparse_inducing_spacing_V = None  # "sparse" only: inducing grid (None = length scale / 4)
# --- GP strategy: pick a preset (selfdriving_afm/gp_presets.py, README "GP presets") ---
#   "original"             first scoring (gradient + exploration x sigma): mostly explores
#   "hybrid"               GP edge-weighted exploration + rule-based edge tracing
#   "pure_gp"              variance reduction (IVR), lambda 0.8, straight segments
#   "pure_gp_smooth"       pure_gp, segments tangent to each other (no sharp turns)  <- best map at 120 V
#   "pure_gp_1V"           pure_gp, 1 V segments (half the waveforms)
#   "pure_gp_smooth_1V"    pure_gp_1V with smooth turns
#   "pure_gp_edges"        pure_gp, lambda 1.0 (uncertainty reduction only at edges)
#   "pure_gp_edges_smooth" pure_gp_edges with smooth turns
from selfdriving_afm.gp_presets import apply_gp_preset
GP_PRESET = "pure_gp_smooth"
apply_gp_preset(cfg, GP_PRESET)
# optional overrides AFTER the preset, e.g.:
# cfg.gp_path.ivr_lambda = 0.8               # pure_gp*: exploration (0) .. exploitation (1)
# cfg.gp_path.max_turn_deg = 90              # *_smooth: gentler curves (min radius ~0.055 V)
cfg.gp_path.max_path_length_V = 60.0           # scan-time budget: total tip travel (V); None = off
cfg.gp_path.gp_n_restarts_optimizer = 0     # 0 = fast (fixed init hyperparams); raise once
                                              # you're tuning against real data for a better fit
#
# --- stopping criteria: several apply at once, whichever fires first wins. Toggle
# any of them off by setting it to None. See README "When does a scan stop?" for
# the full reasoning behind each one.
cfg.gp_path.max_segments = 400              # (1) hard cap, always enforced regardless of the below
                                             #     (tracing uses short 0.2 V steps: the path
                                             #     budget above is the real limit)
cfg.gp_path.max_run_time_s = 1020.0          # (2) TIME: wall-clock cap in seconds, e.g. 120 = 2
                                             #     minutes. Set to None to disable.
cfg.gp_path.target_mean_uncertainty = None  # (3) "good enough": stop once the GP's mean
                                             #     uncertainty over stop_check_extent_V drops to
                                             #     this (same units as z). None disables it --
                                             #     set a value once you know a reasonable noise
                                             #     floor from simulation, e.g. 3.0
cfg.gp_path.stop_check_extent_V = 0.3        # required if target_mean_uncertainty is set
cfg.gp_path.stop_check_grid_n = 30           # resolution of the grid that uncertainty check runs on
cfg.gp_path.min_segments_before_stop_check = 8  # don't even check (3) until this many segments
                                                  # are in, so a lucky-quiet early patch can't
                                                  # end the run prematurely

# --- edge-explorer knobs (only used if MODALITY == "edge_explorer") ---
# (area, first path, exploration spacing and zigzag step range: set by the priors, section 2b)
# a continuous path of straight segments: EXPLORES until it crosses an edge, then FOLLOWS
# that edge (a zigzag across it) until it's closed -- or lost both ways -- then explores
# again; other edges crossed meanwhile are queued and traced next. Closed edges are not
# re-scanned. Only a few ms of computing between segments.
cfg.edge_explorer.initial_path_length_V = 1.0   # the first straight path
cfg.edge_explorer.initial_direction_deg = 0.0   # its direction (0 = +X, 90 = +Y)
cfg.edge_explorer.segment_length_V = 0.5        # step size while exploring
cfg.edge_explorer.follow_recovery_tries = 3      # following: a zigzag step missed the edge (a corner) -> turn
cfg.edge_explorer.follow_recovery_turn_deg = 60.0  #   this much and retry, this many times, before it's lost
cfg.edge_explorer.max_segments_per_feature = 300  # following: safety cap per direction along an edge
# following style: "segments" = zigzag of straight segments across the edge (default);
# "weave" = a smooth sine weaving back and forth across it (no corners, easier on the piezo),
# re-centred and re-aimed on the edge after every waveform (see README "smooth edge following")
cfg.edge_explorer.follow_style = "weave"     # "segments" or "weave"
cfg.edge_explorer.follow_weave_amplitude_V = 0.4   # weave: how far either side of the edge (None = half the
                                                 #   shortest follow step, >= 0.2 V -- wider than the piezo error)
cfg.edge_explorer.follow_weave_wavelength_V = 0.08  # weave: distance along the edge per sine cycle (None = same rule)
cfg.edge_explorer.follow_weave_cycles = 3.0      # weave: sine cycles per waveform (then re-fitted to its crossings)
cfg.edge_explorer.follow_weave_corner_deg = 60.0  # weave: crossings stopped part-way (a corner) -> restart turned this much
cfg.edge_explorer.scan_interior = True          # once an edge closes: parallel lines across its inside (one
                                                 #   waveform) -- measures the top, finds holes (followed too)
cfg.edge_explorer.interior_max_lines = 25       # at most this many lines per feature (their spacing and margin
                                                 #   come from the priors)
cfg.edge_explorer.explore_scale_V = None        # exploring sweeps ~this far apart = smallest feature you
                                                 #   don't want to miss (None = segment_length_V); bigger = faster
                                                 #   (the priors set it to the smallest feature)
cfg.edge_explorer.explore_lookahead_V = 4.0      # each direction is scored this far ahead (None = 4 x
                                                 #   segment_length_V): bigger = heads for big empty regions
cfg.edge_explorer.min_score = 0.3                # when nothing nearby is more unexplored than this, jump to
                                                 #   the nearest unexplored region (0.02 = squeeze into thin gaps)
cfg.edge_explorer.straight_bonus = 0.3           # preference for going straight on: longer sweeps (0.1 = curlier)
cfg.edge_explorer.loop_closure_min_coverage_deg = 300.0  # closed = back at the start, crossings this far round
cfg.edge_explorer.edge_detect.mode = "auto_mad"  # "auto_mad" or "manual" + manual_threshold
cfg.edge_explorer.edge_detect.manual_threshold = None
cfg.edge_explorer.edge_detect.mad_k = 6.0
# stopping: whichever comes first, checked BETWEEN features -- the edge being followed is
# always finished first (it also stops once the whole region has been explored)
cfg.edge_explorer.max_segments = 500
cfg.edge_explorer.max_features = 25             # closed edges
cfg.edge_explorer.max_run_time_s = 1200.0

# --- grid + refine knobs (only used if MODALITY == "grid_refine") ---
# (area and sparse line spacing: set by the priors, section 2b)
# 1. ONE sparse scan of the whole area -> 2. fast inpainting -> 3. find where the data shows
# structure (local height variation with the local tilt removed, vs the noise) -> 4. tiles
# there (+ the insides of closed rings of tiles = features' flat tops); touching tiles = one
# REGION -> 5. one denser scan per region, its pattern from its shape -> 6. repeat at the
# new density inside the refined regions, until the finest spacing. Every scan and every
# move between scans starts and ends at rest (smooth motion, no jumps).
# who generates the scan paths:
cfg.grid_refine.scan_generator = "pyscanner_defined"   # "user_defined" = the package builds every waveform
                                               #   (constant tip speed, a level's tiles in one waveform);
                                               # "pyscanner_defined" = PyScanner's own scan types (raster 2,
                                               #   Lissajous 3, spiral 1) placed with its scan offsets, one
                                               #   do_scan per tile -- run the check in section 3b first
cfg.grid_refine.pyscanner_speed_basis = "peak"  # "pyscanner_defined": "peak" = never faster than tip_velocity_V_s,
                                               #   "mean" = tip_velocity_V_s on average (PyScanner's scans are sinusoidal)
cfg.grid_refine.pyscanner_raster_type = 2      # PyScanner raster_type (2 = sinusoidal)
# 1. sparse scan
cfg.grid_refine.sparse_pattern = "spiral"     # "raster", "lissajous" or "spiral"
cfg.grid_refine.sparse_crosshatch = False      # raster: second pass with lines the other way (a grid);
                                               #   False = ONE scan over the whole area
# 4. refinement
cfg.grid_refine.refine_pattern = "auto"       # "auto" = each region's pattern from its shape: round -> spiral
                                               #   (when shorter than a raster), rectangular -> auto_rect_pattern,
                                               #   irregular -> raster with every line clipped to the region;
                                               #   or force "raster", "lissajous" or "spiral" for every region
cfg.grid_refine.auto_rect_pattern = "raster"  # "auto": rectangular regions -- "raster" or "lissajous"
cfg.grid_refine.merge_tiles = True            # touching tiles (sides or corners) = one region, one scan
cfg.grid_refine.fill_interiors = True         # a closed ring of tiles (a feature's outline): scan its inside
                                               #   too -- measures the flat top, finds internal edges. Insides
                                               #   up to the largest feature wide (priors) -- bigger = substrate
cfg.grid_refine.fill_interior_levels = 1      # ... on the first N refinement levels only (later: edges only)
cfg.grid_refine.refine_factor = 3.0           # each level has lines this much closer
cfg.grid_refine.min_spacing_V = None          # final resolution = finest line spacing (None = width / 255,
                                               #   like a 256-line raster)
cfg.grid_refine.tile_size_lines = 2.0         # tile width, in line spacings of the level before (at most
                                               #   half the smallest feature -- priors)
cfg.grid_refine.raster_direction = "auto"     # "auto" (lines cross the edges), "x", "y" or "alternate"
cfg.grid_refine.batch_tiles = True            # "user_defined": all regions of a level in ONE waveform
                                               #   ("pyscanner_defined": one do_scan per region)
cfg.grid_refine.lissajous_phi_rad = None      # None = 0 (covers most evenly)
# 3. detection
cfg.grid_refine.detect_k = 5.0                # flag where local variation > k x noise -- higher = fewer tiles
cfg.grid_refine.detect_threshold = None       # or a fixed threshold (units of AI0) instead of the auto one
cfg.grid_refine.detect_on = "reconstruction"        # "measured" = decide from the measured pixels only (path independent
                                               #   of any reconstruction); "reconstruction" = inpaint first, decide
                                               #   on the inpainted image (finds structure predicted between lines)
cfg.grid_refine.detect_reconstruction = "linear"  # inpainting for "reconstruction": "linear", "fast",
                                               #   "compressed_sensing", "gp" or "live_view" (= cfg.live_view.reconstruction;
                                               #   CS/GP use the live-view parameters above and take longer per level)
# smooth motion + stopping
cfg.grid_refine.ease_time_s = 0.005           # speed-up / slow-down time at every scan start and end
cfg.grid_refine.max_levels = 2
cfg.grid_refine.max_run_time_s = None

# --- readback (real hardware only) -- Z/deflection input; there's no AO for Z, X/Y
# are the only outputs (AO_X_CONTROL/AO_Y_CONTROL in backends/pyscanner_backend.py) ---
cfg.readback.ai_channel = "AI0"  # CONFIRM ON HARDWARE which AI your deflection signal is wired to
cfg.readback.mode = "raw"  # "raw" (per-sample, CONFIRM ON HARDWARE) or "image" (binned)
# PyScanner fast rasters (grid_refine): the AI record lags/leads the path in time, so
# trace and retrace show every feature twice along the fast axis. "fit" = fit the
# shift on every fast raster (trace and retrace must agree) and re-place Z; "fixed" =
# always fast_raster_delay_s; "off" = none. fast_raster_delay_s is also the starting
# estimate for "fit" (+0.127 s measured on 2026-10-06)
cfg.readback.fast_raster_delay = "fit"
cfg.readback.fast_raster_delay_s = 0.127

# --- Asylum Research / Igor (section 3a and around every run cell) ---
cfg.asylum.controller = "cypher"     # X/Y loops read "Cypher.Input.A/B" ("arc": "Arc.Input.A/B")
cfg.asylum.crosspoint = {"InA": "BNCIn0", "InB": "BNCIn1"}   # FPGA AO0 / AO1 come in on BNCIn0 / BNCIn1
cfg.asylum.lock_crosspoint = True    # lock them, so a mode change doesn't re-route them
cfg.asylum.center_first = True       # setup: Go There (GoForce_1, spot 0) before handing X/Y to the FPGA
cfg.asylum.center_transit_s = 2.0     # time given to get there
cfg.asylum.engage_wait_s = 5.0       # time SimpleEngageMe gets to reach the surface
cfg.asylum.withdraw_wait_s = 2.0

# --- X/Y position sensors (optional): the real tip position, recorded with Z ---
# The controller's XSensor / YSensor are copied to Cypher BNCOut0 / BNCOut1 (a PIDS loop each
# -> a free DAC -> the BNC, locked), wired to PyScanner AI1 / AI2. The setup cell (3a) routes
# them and CALIBRATES them (tip withdrawn: a 3 x 3 grid of drive positions; AI V -> sensor V
# -> um with Igor's XLVDTSens / XLVDTOffset -> drive V); every run cell re-checks the routing
# (an Igor scan resets it). With a calibration, every image is built from the measured
# positions (piezo errors gone) and the axes are in um. Needs readback.mode == "raw".
cfg.position_sensors.enabled = True
cfg.position_sensors.x_ai_channel = "AI1"   # BNCOut0 (XSensor) wired here
cfg.position_sensors.y_ai_channel = "AI2"   # BNCOut1 (YSensor) wired here
cfg.position_sensors.x_loop = 3              # Cypher PIDS loops doing the copy (not 0/1: X/Y follow the FPGA)
cfg.position_sensors.y_loop = 4
cfg.position_sensors.x_bnc = "BNCOut0"
cfg.position_sensors.y_bnc = "BNCOut1"
cfg.position_sensors.calibration_span_V = 5.0    # calibration grid: +- this, in drive volts
cfg.position_sensors.calibration_settle_s = 0.3  # wait at each point before recording
cfg.position_sensors.x_m_per_V = None        # sensor V -> m; None = Igor's XLVDTSens (3.83 um/V on 2026-10-08)
cfg.position_sensors.y_m_per_V = None        #                      YLVDTSens (3.98 um/V)
cfg.position_sensors.smooth_samples = 0      # moving average over N samples (sensor noise); 0 = none
cfg.position_sensors.divide_output = True   # True: OutC/OutD = sensor / output_divider, so sensors past
cfg.position_sensors.output_divider = 2.0    #   +-10 V still fit the BNC / AI range (calibration absorbs it)
# with a good calibration (the setup cell prints "calibration NOT OK" otherwise -> all of these off):
cfg.position_sensors.plan_on_sensors = True   # every mode decides where to scan next from the MEASURED positions
cfg.position_sensors.correct_linear = True    # (a) waveforms sent through the inverse drive map (scale, cross-talk)
cfg.position_sensors.correct_offset = True    # (b) each waveform's measured end error shifts the next ones
cfg.position_sensors.correction_gain = 0.5    #     (b) fraction of a new error taken each time (1 = all of it)
cfg.position_sensors.correction_max_V = 1.0   #     (b) never more than this (drive volts)

# --- Z signal on a Cypher BNC (cfg.z_signal; asylum.AsylumControl.route_z, setup step 3d) ---
# PyScanner's AI0 (cfg.readback.ai_channel) stays cabled to z_signal.bnc; the setup copies the
# chosen Z signal there through a Cypher loop and checks (tip withdrawn) that AI0 reads it
cfg.z_signal.route = True            # True: AI0's cable on BNCOut2, signal chosen below
cfg.z_signal.source = "ZSensor"        # "ZSensor" (Z LVDT) or the Z piezo's Cypher signal name (CONFIRM)
cfg.z_signal.loop = 5                  # Cypher PIDS loop doing the copy (3/4: X/Y sensors)
cfg.z_signal.dac = "OutFastA"          # preferred DAC (another free one if taken; OutA/B = Igor's loops 0/1)
cfg.z_signal.bnc = "BNCOut2"
cfg.z_signal.output_divider = 1.0      # BNC = signal / this (BNC / AI stop at +-10 V)
cfg.z_signal.invert = True             # flip the sign in software: ZSensor reads opposite to the height
cfg.z_signal.units = "nm"              # "nm": Z converted to nm as it comes in (needs route + ZSensor's
                                       #   ZLVDTSens): plots, saved z AND thresholds in nm, e.g.
                                       #   cfg.edge_explorer.edge_detect.manual_threshold = 100.0  # nm

# --- safety ---
cfg.safety.max_amplitude_V = 10.0   # spirals are moved/shrunk to stay inside this; circles stop at it
cfg.safety.max_samples_per_waveform = 4_000_000  # longer waveforms go to PyScanner in pieces of this size
                                                   # (3.8 M worked on hardware, 8.5 M did not)
cfg.safety.clamp_mode = "raise"  # keep "raise" for real runs -- see README

# --- live view (wall-clock throttled -- see LiveViewConfig) ---
cfg.live_view.min_redraw_interval_s = 0.25
cfg.live_view.min_reconstruction_interval_s = 1.0
cfg.live_view.figsize = (13.5, 4.0)  # inches, 3 columns -- shrink this (e.g. (10, 3)) if it's
                                       # still wider than your notebook window
# --- reconstruction (edge-follow live view: col 1 = scan path, col 2 = binned AI0
# exactly like PyScanner's image_AI0, col 3 = col 2's empty pixels filled in) ---
cfg.live_view.reconstruction = "compressed_sensing"  # "compressed_sensing", "gp", "linear", "fast" or "interpolated"
                                          #   "linear" = interpolation between ALL measured pixels (the old
                                          #   "interpolated" only used 4000 subsampled points) -- best on steps
                                                     #   ("fast": local average -- no CS ghosts on sparse grid_refine data)
cfg.live_view.n_image_pixels = 200     # binned/reconstructed image size -- match PyScanner's N_image_pixels
# compressed sensing (image assumed sparse in the 2D DCT; ~0.2 s per update)
cfg.live_view.cs_lambda = 0.005       # higher = smoother, lower = sticks closer to the measured pixels (was 0.5)
cfg.live_view.cs_iterations = 500      # more = more converged, slower (~0.2 s per update at 500)
cfg.live_view.cs_detrend = True        # remove the sample tilt (best-fit plane) before CS -- sharper features
# GP (RBF + noise kernel; ~1 s per update with gp_optimize=False)
cfg.live_view.gp_kernel = "matern52"   # "matern52" (as in the SS-KPFM paper, no ringing at edges) or "rbf"
cfg.live_view.gp_max_points = 1500     # measured pixels the GP trains on -- the speed knob (cost ~ n^3)
cfg.live_view.gp_length_scale_frac = 0.05  # feature size, as a fraction of the image width
cfg.live_view.gp_optimize = False      # True = fit the kernel each update: better, but seconds-to-tens-of-
                                       # seconds per update -- raise min_reconstruction_interval_s if you do
LIVE_VIEW_EXTENT_V = 0.45  # starting/fallback view size ONLY, for before any data exists
                            # (and the seed for cell 5's saved reconstruction if the run
                            # somehow has zero points) -- once the scan has points, every
                            # panel's axes auto-fit to wherever the tip has actually been
                            # instead (see viz.bounds_from_history), so this doesn't need
                            # to match your actual search/path radii


## Overrides for today's GP runs

In [ ]:
# --- GP run (Fig. 4) ---
GP_PRESET = "pure_gp_smooth"
apply_gp_preset(cfg, GP_PRESET)
cfg.gp_path.max_path_length_V = 120.0     # tip-travel budget (V)
cfg.gp_path.max_segments = 600
cfg.gp_path.max_run_time_s = 1500.0
# end-of-waveform dwell at rest (s): absorbs PyScanner's short AI record (set from the check below)
TAIL_DWELL_S = 0.0   # readback check 2026-11-09: <= 0.5 s waveforms come back whole; 2 s ones lose 2-52 ms
# priors in nm (sensor frame): hexagons / disc ~4 um wide over a ~28 um field
PRIORS_NM = dict(feature_min_nm=3000, feature_max_nm=5000, area_half_width_nm=13000, area_center_nm=(0, 0))
print("preset", GP_PRESET, "| budget", cfg.gp_path.max_path_length_V, "V | dwell", TAIL_DWELL_S, "s")

In [ ]:
if USE_HARDWARE:
    backend = PyScannerBackend(vi, cfg.safety, cfg.readback, position_sensors=cfg.position_sensors)
    ground_truth_fn = None
else:
    backend = SimulateBackend(noise_std=0.001, seed=1)
    ground_truth_fn = default_topography


In [ ]:
RUN_SETUP = True   # <-- False to skip (e.g. already set up and engaged in this session)

if RUN_SETUP or "afm" not in globals() or afm.backend is not backend:
    afm = asylum.AsylumControl(igor, backend, cfg)   # the executor joins in 3c (afm.run passes it on)
if RUN_SETUP:
    afm.setup()

## Priors (non-interactive)

In [ ]:
from selfdriving_afm import priors

# --- the defaults offered when asked (Enter keeps them) ---
PRIOR_DEFAULTS = priors.ScanPriors(
    feature_min_V=0.5,          # smallest expected feature: its narrowest width (V)
    feature_max_V=2.0,          # largest expected feature (V)
    area_half_width_V=4.0,      # max area to explore / discover: a square of this half-width (V)...
    area_center_V=(0.0, 0.0),   # ...centered here (V)
)
# with the position sensors on: asked in nm (sensor frame). None = PRIOR_DEFAULTS converted with the
# setup's calibration; or e.g. dict(feature_min_nm=2000, feature_max_nm=6000, area_half_width_nm=15000,
# area_center_nm=(0, 0))  -- the centre in the sensors' coordinates (the plots' um axes x 1000)
PRIOR_DEFAULTS_NM = PRIORS_NM
ASK_PRIORS = False              # False = use the defaults without asking

PRIORS = priors.ask_for_run(cfg, PRIOR_DEFAULTS, PRIOR_DEFAULTS_NM, sensor_cal=getattr(afm, "sensor_cal", None),
                            interactive=ASK_PRIORS)
# edge detection (edge_explorer, edge_follow): the on-feature threshold, height above the substrate --
# in nm when Z comes in as nm (cfg.z_signal.units = "nm" + ZSensor routed in 3a), else in V. Enter keeps
# cell 5's settings (auto = 6 x the roughness, or the gap to the features); a number = that fixed
# threshold for both edge modes; 'auto' = automatic for both
PRIORS = priors.ask_edge_threshold(cfg, PRIORS, z_info=getattr(backend, "z_signal_info", None),
                                   interactive=ASK_PRIORS, modality=MODALITY)
print("Priors:", PRIORS)
for note in priors.apply(cfg, PRIORS):   # sets every mode's area + size parameters in cfg
    print(" ", note)
LIVE_VIEW_EXTENT_V = PRIORS.area_half_width_V
# the live view always shows this whole area (cfg.live_view.fixed_bounds_V, set by the priors);
# cfg.live_view.fixed_bounds_V = None here to let it grow with the scanned area instead


In [ ]:
# executor + planner + live view for MODALITY. This is the SAME code Simulation.ipynb
# runs (selfdriving_afm/modes.py) -- only the backend differs -- so a mode behaves here
# exactly as it did in simulation. New modes are added there once, for both notebooks.
session = modes.build(MODALITY, cfg, backend, live=True, live_extent=LIVE_VIEW_EXTENT_V,
                      ground_truth_fn=ground_truth_fn, priors=PRIORS)
executor, planner, live = session.executor, session.planner, session.live

# Building the live view above pushed an initial (empty) frame into THIS cell's own
# output, just to confirm the view was built correctly. The actual live view while
# scanning shows up in cell 4's OWN output instead (it gets replaced in place there,
# once per redraw) -- not here -- since each cell has its own separate output area.
# See viz._LiveViewBase._redraw's docstring for why it's done this way rather than
# via the interactive ipympl canvas's own redraw.
afm.executor = executor   # the setup's calibration and Z routing reach it through the backend / afm.run


## Readback-length check (tip withdrawn)

Out-and-back cosine waveforms of several durations from where the outputs are; how many AI samples come back vs sent.

In [ ]:
afm.withdraw()
RB_DURATIONS_S = [0.017, 0.05, 0.15, 0.5, 2.0]
RB_AMPLITUDE_V = 0.5
rate = cfg.velocity.sample_rate_hz
rb_log = []
x0, y0 = backend.last_out if backend.last_out is not None else (0.0, 0.0)
for T in RB_DURATIONS_S:
    for rep in range(3):
        n = int(round(T * rate))
        u = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(n) / (n - 1))
        xs, ys_ = x0 + RB_AMPLITUDE_V * u, y0 + 0 * u
        t1 = time.perf_counter()
        z = backend.measure_path(xs, ys_)
        rb_log.append(dict(T=T, sent=n, got=len(z), missing=n - len(z), wall=time.perf_counter() - t1))
        print(f"T={T:6.3f} s  sent {n:>9,}  recorded {len(z):>9,}  missing {n - len(z):>8,} ({(n - len(z)) / rate * 1e3:6.1f} ms)  wall {rb_log[-1]['wall']:.2f} s")
executor.reset(cfg.execution.start_xy_V)
print("max missing:", max(r["missing"] for r in rb_log) / rate * 1e3, "ms")

In [ ]:
executor.tail_dwell_s = TAIL_DWELL_S
print("executor.tail_dwell_s =", executor.tail_dwell_s)

## Run (engaged; withdraws and returns to (0, 0) at the end)

In [ ]:
# runs the planner from cfg.execution.start_xy_V, finalizes the live view and prints
# the mode's summary (modes.MODES[MODALITY].summary) -- same as in Simulation.ipynb.
# Engaged for the run; withdraws and goes back to (0, 0) smoothly when it ends (also
# if it fails or is interrupted) -- see section 3a
with afm.run(MODALITY, executor):
    result = modes.run(session)
# full scan-by-scan log (spirals, circles, moves): print("\n".join(live.log))


In [ ]:
# trajectory + reconstruction (+ grid_refine maps and tiles) -> OUTPUT_DIR, in the format
# Analyze_SelfDriving_Scan.ipynb reads -- see modes.save
out_prefix = modes.save(session, OUTPUT_DIR, prefix="scan", fallback_extent=LIVE_VIEW_EXTENT_V)


## Equivalent raster: user-defined, drive-corrected, cut only at line ends

In [ ]:
RUN_RASTER = True
from selfdriving_afm import user_raster, reference_raster
RUN_NPZ = f"{out_prefix}.npz"
cfg.grid_refine.raster_turn_radius_V = 0.3     # gentle half-ellipse turnarounds
cfg.grid_refine.ease_time_s = 0.02
cfg.safety.max_samples_per_waveform = executor.safety.max_samples_per_waveform = int(4.0 * cfg.velocity.sample_rate_hz)
_c, _h = PRIORS.area_center_V, PRIORS.area_half_width_V
RASTER_AREA = (_c[0] - _h, _c[0] + _h, _c[1] - _h, _c[1] + _h)   # the GP's area (priors)
RP = user_raster.plan(RUN_NPZ, cfg, executor, area=RASTER_AREA)
print(user_raster.describe(RP)); ok = user_raster.ram_check(RP, cfg)
if RUN_RASTER and ok:
    with afm.run("equivalent raster (user-defined)", executor):
        REF = user_raster.acquire(RP, executor, cfg)
    reference_raster.save_into(RUN_NPZ, REF)
    print("saved into", RUN_NPZ)

In [ ]:
# --- campaign: several GP strategies, each followed by its equivalent raster ---
CAMPAIGN = [("pure_gp_edges_smooth", 240.0), ("hybrid", 240.0), ("pure_gp_smooth", 240.0)]
from selfdriving_afm import user_raster, reference_raster
campaign_out = []
for preset, budget in CAMPAIGN:
    print("=" * 30, preset, budget, "V", "=" * 30)
    apply_gp_preset(cfg, preset)
    cfg.gp_path.max_path_length_V = budget; cfg.gp_path.max_segments = 1200; cfg.gp_path.max_run_time_s = 1500.0
    for note in priors.apply(cfg, PRIORS):
        pass
    session = modes.build(MODALITY, cfg, backend, live=True, live_extent=LIVE_VIEW_EXTENT_V, ground_truth_fn=None, priors=PRIORS)
    executor, planner, live = session.executor, session.planner, session.live
    afm.executor = executor
    executor.tail_dwell_s = 0.0
    with afm.run(f"{MODALITY} {preset} {budget:g} V", executor):
        result = modes.run(session)
    out_prefix = modes.save(session, OUTPUT_DIR, prefix=f"scan_{preset}_{int(budget)}V", fallback_extent=LIVE_VIEW_EXTENT_V)
    RUN_NPZ = f"{out_prefix}.npz"
    _c, _h = PRIORS.area_center_V, PRIORS.area_half_width_V
    RP = user_raster.plan(RUN_NPZ, cfg, executor, area=(_c[0] - _h, _c[0] + _h, _c[1] - _h, _c[1] + _h))
    print(user_raster.describe(RP))
    if user_raster.ram_check(RP, cfg):
        with afm.run("equivalent raster (user-defined)", executor):
            REF = user_raster.acquire(RP, executor, cfg)
        reference_raster.save_into(RUN_NPZ, REF)
    campaign_out.append(RUN_NPZ)
    plt.close("all")
print("campaign done:", campaign_out)

In [ ]:
# --- reload the 2026-11-09 fixes (TV map for GP modes) + dense reference of the area, same sensor frame ---
import importlib, gc, json as _json
from selfdriving_afm import reconstruction, viz, modes, user_raster, reference_raster
for _m in (reconstruction, viz, modes, user_raster):
    importlib.reload(_m)
cfg.live_view.reconstruction = "tv"; cfg.live_view.tv_mu = 0.3; cfg.live_view.tv_iterations = 1500
cfg.live_view.gp_view = "reconstruction"; cfg.live_view.live_tv_iterations = 300
for _n in ("session", "planner", "live", "REF", "result"):
    globals().pop(_n, None)
gc.collect()
_c, _h = PRIORS.area_center_V, PRIORS.area_half_width_V
AREA = (_c[0] - _h, _c[0] + _h, _c[1] - _h, _c[1] + _h)
cfg.readback.store_every = 25          # block means of 25 samples (0.75 nm of path) -- RAM
RPd = user_raster.plan(campaign_out[0], cfg, executor, n_lines=150, area=AREA)
print(user_raster.describe(RPd)); ok = user_raster.ram_check(RPd, cfg)
if ok:
    with afm.run("dense reference raster (150 lines)", executor):
        DENSE = user_raster.acquire(RPd, executor, cfg)
    cal = afm.sensor_cal
    DENSE_NPZ = OUTPUT_DIR / f"dense_reference_150lines_{time.strftime('%Y%m%d_%H%M%S')}.npz"
    np.savez(DENSE_NPZ, area=np.array(AREA), um_per_V=np.array(cal.um_per_V), um_at_0V=np.array([cal.x.b * 1e6, cal.y.b * 1e6]),
             **{k: v for k, v in DENSE.items() if isinstance(v, np.ndarray)},
             band_log=np.array(_json.dumps(DENSE["band_log"])), wall_time_s=DENSE["wall_time_s"], lines_done=DENSE["lines_done"])
    print("saved", DENSE_NPZ)
cfg.readback.store_every = 1
del DENSE; gc.collect()

In [ ]:
# --- hero run for Fig. 4: pure_gp_edges_smooth, 240 V, TV live view, + its equivalent raster ---
HERO = []
for preset, budget in [("pure_gp_edges_smooth", 240.0)]:
    apply_gp_preset(cfg, preset)
    cfg.gp_path.max_path_length_V = budget; cfg.gp_path.max_segments = 1200; cfg.gp_path.max_run_time_s = 1500.0
    for note in priors.apply(cfg, PRIORS):
        pass
    session = modes.build(MODALITY, cfg, backend, live=True, live_extent=LIVE_VIEW_EXTENT_V, ground_truth_fn=None, priors=PRIORS)
    executor, planner, live = session.executor, session.planner, session.live
    afm.executor = executor; executor.tail_dwell_s = 0.0
    t_wall = time.time()
    with afm.run(f"{MODALITY} {preset} {budget:g} V (hero)", executor):
        result = modes.run(session)
    t_wall = time.time() - t_wall
    out_prefix = modes.save(session, OUTPUT_DIR, prefix=f"scan_hero_{preset}_{int(budget)}V", fallback_extent=LIVE_VIEW_EXTENT_V,
                            extra=dict(run_wall_time_s=np.array(t_wall)))
    RUN_NPZ = f"{out_prefix}.npz"
    RP = user_raster.plan(RUN_NPZ, cfg, executor, area=AREA)
    print(user_raster.describe(RP))
    if user_raster.ram_check(RP, cfg):
        with afm.run("equivalent raster (user-defined)", executor):
            REF = user_raster.acquire(RP, executor, cfg)
        reference_raster.save_into(RUN_NPZ, REF)
    HERO.append(RUN_NPZ)
print("hero done:", HERO, "run wall time", round(t_wall, 1), "s")

In [ ]:
# --- fine ground-truth raster: 256 lines over the area (~100 nm apart), same sensor frame ---
import gc, json as _json
gc.collect()
cfg.readback.store_every = 25
RPg = user_raster.plan(campaign_out[0], cfg, executor, n_lines=256, area=AREA)
print(user_raster.describe(RPg)); ok = user_raster.ram_check(RPg, cfg)
if ok:
    with afm.run("ground-truth raster (256 lines)", executor):
        GT = user_raster.acquire(RPg, executor, cfg)
    cal = afm.sensor_cal
    GT_NPZ = OUTPUT_DIR / f"ground_truth_256lines_{time.strftime('%Y%m%d_%H%M%S')}.npz"
    np.savez(GT_NPZ, area=np.array(AREA), um_per_V=np.array(cal.um_per_V), um_at_0V=np.array([cal.x.b * 1e6, cal.y.b * 1e6]),
             **{k: v for k, v in GT.items() if isinstance(v, np.ndarray)},
             band_log=np.array(_json.dumps(GT["band_log"])), wall_time_s=GT["wall_time_s"], lines_done=GT["lines_done"])
    print("saved", GT_NPZ)
    del GT
cfg.readback.store_every = 1
gc.collect()

In [ ]:
# --- exploration/exploitation sweep: pure_gp_smooth, ivr_lambda 0 (explore) -> 1 (edges), 240 V each, + rasters ---
SWEEP = [0.0, 0.25, 0.5, 0.75, 1.0]
SWEEP_OUT = []
for lam in SWEEP:
    print("=" * 30, "ivr_lambda", lam, "=" * 30)
    apply_gp_preset(cfg, "pure_gp_smooth"); cfg.gp_path.ivr_lambda = lam
    cfg.gp_path.max_path_length_V = 240.0; cfg.gp_path.max_segments = 1200; cfg.gp_path.max_run_time_s = 1500.0
    for note in priors.apply(cfg, PRIORS):
        pass
    session = modes.build(MODALITY, cfg, backend, live=True, live_extent=LIVE_VIEW_EXTENT_V, ground_truth_fn=None, priors=PRIORS)
    executor, planner, live = session.executor, session.planner, session.live
    afm.executor = executor; executor.tail_dwell_s = 0.0
    t_wall = time.time()
    with afm.run(f"gp_path lambda {lam}", executor):
        result = modes.run(session)
    t_wall = time.time() - t_wall
    tag = f"scan_sweep_lam{int(round(lam * 100)):03d}_pure_gp_smooth_240V"
    out_prefix = modes.save(session, OUTPUT_DIR, prefix=tag, fallback_extent=LIVE_VIEW_EXTENT_V,
                            extra=dict(run_wall_time_s=np.array(t_wall), ivr_lambda=np.array(lam)))
    RUN_NPZ = f"{out_prefix}.npz"
    RP = user_raster.plan(RUN_NPZ, cfg, executor, area=AREA)
    if user_raster.ram_check(RP, cfg):
        with afm.run("equivalent raster (user-defined)", executor):
            REF = user_raster.acquire(RP, executor, cfg)
        reference_raster.save_into(RUN_NPZ, REF)
    SWEEP_OUT.append(RUN_NPZ)
    print("DONE lambda", lam, "wall", round(t_wall, 1), "s ->", RUN_NPZ)
    for _n in ("session", "planner", "live", "REF", "result"):
        globals().pop(_n, None)
    gc.collect(); plt.close("all")
print("sweep done:", SWEEP_OUT)

In [ ]:
# --- rasters the sweep skipped (RAM check failed): re-acquire for lambda 0, 0.5, 0.75 ---
import zipfile, psutil
MISSING = [p for p in SWEEP_OUT if not any(n.startswith("raster_") for n in zipfile.ZipFile(p).namelist())]
print("missing rasters:", MISSING)
for RUN_NPZ in MISSING:
    for _n in ("session", "planner", "live", "REF", "result"):
        globals().pop(_n, None)
    gc.collect()
    print(f"RAM free {psutil.virtual_memory().available / 1e9:.1f} GB; python {psutil.Process().memory_info().rss / 1e9:.1f} GB")
    executor.history.clear(); gc.collect()
    RP = user_raster.plan(RUN_NPZ, cfg, executor, area=AREA)
    print(user_raster.describe(RP))
    if user_raster.ram_check(RP, cfg, fraction=0.7):
        with afm.run("equivalent raster (user-defined)", executor):
            REF = user_raster.acquire(RP, executor, cfg)
        reference_raster.save_into(RUN_NPZ, REF)
        del REF; gc.collect()
    else:
        print("STILL NOT ENOUGH RAM for", RUN_NPZ)
print("done")

In [ ]:
# --- clean re-runs (no CPU contention) of lambda 0 and 0.25, + rasters ---
def gp_run(lam, budget=240.0, preset="pure_gp_smooth", tag_prefix="scan_rerun"):
    global session, executor, planner, live
    for _n in ("session", "planner", "live", "REF", "result"):
        globals().pop(_n, None)
    gc.collect()
    apply_gp_preset(cfg, preset); cfg.gp_path.ivr_lambda = lam
    cfg.gp_path.max_path_length_V = budget; cfg.gp_path.max_segments = 1200; cfg.gp_path.max_run_time_s = 1500.0
    for note in priors.apply(cfg, PRIORS):
        pass
    session = modes.build(MODALITY, cfg, backend, live=True, live_extent=LIVE_VIEW_EXTENT_V, ground_truth_fn=None, priors=PRIORS)
    executor, planner, live = session.executor, session.planner, session.live
    afm.executor = executor; executor.tail_dwell_s = 0.0
    t_wall = time.time()
    with afm.run(f"gp_path {preset} lambda {lam}", executor):
        modes.run(session)
    t_wall = time.time() - t_wall
    tag = f"{tag_prefix}_lam{int(round(lam * 100)):03d}_{preset}_{int(budget)}V"
    out_prefix = modes.save(session, OUTPUT_DIR, prefix=tag, fallback_extent=LIVE_VIEW_EXTENT_V,
                            extra=dict(run_wall_time_s=np.array(t_wall), ivr_lambda=np.array(lam)))
    RUN_NPZ = f"{out_prefix}.npz"
    session.live = None; executor.on_step = executor.on_before_step = None
    gc.collect(); plt.close("all")
    RP = user_raster.plan(RUN_NPZ, cfg, executor, area=AREA)
    if user_raster.ram_check(RP, cfg, fraction=0.7):
        with afm.run("equivalent raster (user-defined)", executor):
            REF_ = user_raster.acquire(RP, executor, cfg)
        reference_raster.save_into(RUN_NPZ, REF_); del REF_
    else:
        print("RASTER SKIPPED (RAM) for", RUN_NPZ)
    gc.collect()
    print("DONE", preset, "lambda", lam, "wall", round(t_wall, 1), "s ->", RUN_NPZ)
    return RUN_NPZ

RERUN_OUT = [gp_run(0.0), gp_run(0.25)]
print("reruns done:", RERUN_OUT)

In [ ]:
# --- hybrid: GP (IVR) between features -> edge tracker on them (gp_edge_explorer), + plain L2 for comparison ---
import importlib, copy as _copy
from selfdriving_afm import reconstruction, viz
from selfdriving_afm.planners import gp_edge_explorer_planner
for _m in (reconstruction, gp_edge_explorer_planner, viz, modes):
    importlib.reload(_m)
EE0 = _copy.deepcopy(cfg.edge_explorer)          # the run notebook's L2 settings (Fig. 3), priors applied below
def edge_run(modality, lam=0.8, tag_prefix="scan_hybrid"):
    global session, executor, planner, live
    for _n in ("session", "planner", "live", "REF", "result"):
        globals().pop(_n, None)
    gc.collect()
    apply_gp_preset(cfg, "pure_gp_smooth"); cfg.gp_path.ivr_lambda = lam
    cfg.edge_explorer = _copy.deepcopy(EE0)
    for note in priors.apply(cfg, PRIORS):
        pass
    ee = cfg.edge_explorer
    if modality == "gp_edge_explorer":
        if ee.follow_segment_length_V is None: ee.follow_segment_length_V = 0.5 * ee.segment_length_V
        if ee.explore_scale_V is None: ee.explore_scale_V = ee.segment_length_V
        ee.segment_length_V = 1.0                       # GP exploring steps (tracing unchanged)
    ee.stop_after_quiet_sweeps = 1.0; ee.max_segments = 2000; ee.max_run_time_s = 1500.0
    print(modality, "| explore step", ee.segment_length_V, "follow", ee.follow_segment_length_V, ee.follow_segment_min_V, ee.follow_segment_max_V,
          "| explore scale", ee.explore_scale_V, "| lambda", cfg.gp_path.ivr_lambda)
    session = modes.build(modality, cfg, backend, live=True, live_extent=LIVE_VIEW_EXTENT_V, ground_truth_fn=None, priors=PRIORS)
    executor, planner, live = session.executor, session.planner, session.live
    afm.executor = executor; executor.tail_dwell_s = 0.0
    t_wall = time.time()
    with afm.run(modality, executor):
        modes.run(session)
    t_wall = time.time() - t_wall
    out_prefix = modes.save(session, OUTPUT_DIR, prefix=tag_prefix, fallback_extent=LIVE_VIEW_EXTENT_V,
                            extra=dict(run_wall_time_s=np.array(t_wall), ivr_lambda=np.array(lam)))
    RUN_NPZ = f"{out_prefix}.npz"
    session.live = None; executor.on_step = executor.on_before_step = None
    gc.collect(); plt.close("all")
    RP = user_raster.plan(RUN_NPZ, cfg, executor, area=AREA)
    if user_raster.ram_check(RP, cfg, fraction=0.7):
        with afm.run("equivalent raster (user-defined)", executor):
            REF_ = user_raster.acquire(RP, executor, cfg)
        reference_raster.save_into(RUN_NPZ, REF_); del REF_
    else:
        print("RASTER SKIPPED (RAM) for", RUN_NPZ)
    gc.collect()
    print("DONE", modality, "wall", round(t_wall, 1), "s ->", RUN_NPZ)
    return RUN_NPZ

HYB_OUT = [edge_run("gp_edge_explorer", 0.8, "scan_hybrid_gpL3_edgeL2"), edge_run("edge_explorer", 0.8, "scan_L2_contour")]
cfg.edge_explorer = _copy.deepcopy(EE0)
print("hybrid done:", HYB_OUT)

In [ ]:
# --- hybrid re-run: the GP explores only (lambda 0 -- the tracer handles edges), 2 quiet sweeps as in L2 (Fig. 3) ---
def edge_run2(modality, lam, tag_prefix, quiet=2.0):
    global session, executor, planner, live
    for _n in ("session", "planner", "live", "REF", "result"):
        globals().pop(_n, None)
    gc.collect()
    apply_gp_preset(cfg, "pure_gp_smooth"); cfg.gp_path.ivr_lambda = lam
    cfg.edge_explorer = _copy.deepcopy(EE0)
    for note in priors.apply(cfg, PRIORS):
        pass
    ee = cfg.edge_explorer
    if modality == "gp_edge_explorer":
        if ee.follow_segment_length_V is None: ee.follow_segment_length_V = 0.5 * ee.segment_length_V
        if ee.explore_scale_V is None: ee.explore_scale_V = ee.segment_length_V
        ee.segment_length_V = 1.0
    ee.stop_after_quiet_sweeps = quiet; ee.max_segments = 2000; ee.max_run_time_s = 1500.0
    session = modes.build(modality, cfg, backend, live=True, live_extent=LIVE_VIEW_EXTENT_V, ground_truth_fn=None, priors=PRIORS)
    executor, planner, live = session.executor, session.planner, session.live
    afm.executor = executor; executor.tail_dwell_s = 0.0
    t_wall = time.time()
    with afm.run(modality, executor):
        modes.run(session)
    t_wall = time.time() - t_wall
    print("STOP REASON:", getattr(planner, "stop_reason", None))
    out_prefix = modes.save(session, OUTPUT_DIR, prefix=tag_prefix, fallback_extent=LIVE_VIEW_EXTENT_V,
                            extra=dict(run_wall_time_s=np.array(t_wall), ivr_lambda=np.array(lam),
                                       stop_reason=np.array(str(getattr(planner, "stop_reason", "")))))
    RUN_NPZ = f"{out_prefix}.npz"
    session.live = None; executor.on_step = executor.on_before_step = None
    executor.history.clear(); executor._last_full_step = None     # saved already: free the RAM for the raster
    gc.collect(); plt.close("all")
    RP = user_raster.plan(RUN_NPZ, cfg, executor, area=AREA)
    if user_raster.ram_check(RP, cfg, fraction=0.7):
        with afm.run("equivalent raster (user-defined)", executor):
            REF_ = user_raster.acquire(RP, executor, cfg)
        reference_raster.save_into(RUN_NPZ, REF_); del REF_
    else:
        print("RASTER SKIPPED (RAM) for", RUN_NPZ)
    gc.collect()
    print("DONE", modality, "wall", round(t_wall, 1), "s ->", RUN_NPZ)
    return RUN_NPZ

HYB2 = edge_run2("gp_edge_explorer", 0.0, "scan_hybrid_lam000_gpL3_edgeL2")
cfg.edge_explorer = _copy.deepcopy(EE0)
print("hybrid (lambda 0) done:", HYB2)

In [ ]:
# --- hybrid, run until all 5 objects are closed: GP (lambda 0) explores, the edge tracker traces, earlier crossings re-checked ---
from selfdriving_afm.planners import gp_edge_explorer_planner
for _m in (gp_edge_explorer_planner, modes):
    importlib.reload(_m)
_old_apply = priors.apply
def edge_run3(lam=0.0, tag_prefix="scan_hybrid5_lam000_gpL3_edgeL2", max_features=5, quiet=3.0):
    EE_run = _copy.deepcopy(EE0)
    globals()["EE0_saved"] = EE0
    return edge_run2_mf("gp_edge_explorer", lam, tag_prefix, quiet, max_features)
def edge_run2_mf(modality, lam, tag_prefix, quiet, max_features):
    # edge_run2 with max_features set (stop once this many loops are closed)
    _orig = edge_run2.__globals__["priors"].apply
    def _apply(cfg_, P_):
        notes = _orig(cfg_, P_)
        cfg_.edge_explorer.max_features = max_features
        return notes
    edge_run2.__globals__["priors"].apply = _apply
    try:
        return edge_run2(modality, lam, tag_prefix, quiet=quiet)
    finally:
        edge_run2.__globals__["priors"].apply = _orig
HYB5 = edge_run3()
cfg.edge_explorer = _copy.deepcopy(EE0)
print("hybrid (all 5) done:", HYB5)

In [ ]:
# --- hybrid with the Fig. 3 tracer: follow steps / closure / interiors as the 1-4 um priors set them ---
_nmV = PRIORS.entered["nm_per_V"] if getattr(PRIORS, "entered", None) else 1875.75
P_FIG3 = priors.ScanPriors(feature_min_V=1000 / _nmV, feature_max_V=4000 / _nmV,
                           area_half_width_V=PRIORS.area_half_width_V, area_center_V=PRIORS.area_center_V)
_tmp = _copy.deepcopy(cfg); _tmp.edge_explorer = _copy.deepcopy(EE0); priors.apply(_tmp, P_FIG3)
FOLLOW_FIG3 = {k: getattr(_tmp.edge_explorer, k) for k in ("follow_segment_length_V", "follow_segment_min_V", "follow_segment_max_V",
                                                           "interior_line_spacing_V", "interior_margin_V", "loop_closure_tolerance_V")}
print("Fig. 3 tracer settings:", {k: round(v, 3) if isinstance(v, float) else v for k, v in FOLLOW_FIG3.items()})
_orig_apply = priors.apply
def _apply_fig3(cfg_, P_):
    notes = _orig_apply(cfg_, P_)
    for k, v in FOLLOW_FIG3.items():
        setattr(cfg_.edge_explorer, k, v)
    cfg_.edge_explorer.max_features = 5
    return notes
priors.apply = _apply_fig3
try:
    HYB_FIG3 = edge_run2("gp_edge_explorer", 0.0, "scan_hybrid5fig3_lam000_gpL3_edgeL2", quiet=3.0)
finally:
    priors.apply = _orig_apply
cfg.edge_explorer = _copy.deepcopy(EE0)
print("hybrid (Fig. 3 tracer) done:", HYB_FIG3)

In [ ]:
# --- edge-focused GP: lambda 1, interest power gamma 2 (best in simulation), 240 V, + raster ---
from selfdriving_afm.planners import gp_path_planner
for _m in (gp_path_planner, gp_edge_explorer_planner, modes):
    importlib.reload(_m)
_apply_preset0 = apply_gp_preset
def _preset_gamma(cfg_, name):
    out = _apply_preset0(cfg_, name)
    cfg_.gp_path.ivr_interest_power = 2.0
    return out
apply_gp_preset = _preset_gamma
try:
    G2 = gp_run(1.0, tag_prefix="scan_edgefocus_gamma2")
finally:
    apply_gp_preset = _apply_preset0
    cfg.gp_path.ivr_interest_power = 1.0
print("gamma 2 done:", G2)

In [ ]:
# --- hybrid re-run: Fig. 3 tracer + corner recovery 5 x 45 deg (was 3 x 60), re-check every 4 choices (was 8) ---
from selfdriving_afm.planners import gp_path_planner, gp_edge_explorer_planner
for _m in (gp_path_planner, gp_edge_explorer_planner, modes):
    importlib.reload(_m)
modes.GPEdgeExplorerPlanner.RECHECK_EVERY = 4
_orig_apply = priors.apply
def _apply_fig3b(cfg_, P_):
    notes = _orig_apply(cfg_, P_)
    for k, v in FOLLOW_FIG3.items():
        setattr(cfg_.edge_explorer, k, v)
    cfg_.edge_explorer.max_features = 5
    cfg_.edge_explorer.follow_recovery_tries = 5
    cfg_.edge_explorer.follow_recovery_turn_deg = 45.0
    return notes
priors.apply = _apply_fig3b
try:
    HYB_FIG3B = edge_run2("gp_edge_explorer", 0.0, "scan_hybrid5fig3b_lam000_gpL3_edgeL2", quiet=3.0)
finally:
    priors.apply = _orig_apply
cfg.edge_explorer = _copy.deepcopy(EE0)
try:
    pl = planner
    for f in pl.features:
        print("feature", f.id, "closed", f.closed_loop, getattr(f, "closed_by", ""), "lost", f.edge_lost, "legs", [len(l) for l in f.legs], "merged", f.merged_ids)
    print("choices:", collections.Counter(pl.choices) if "collections" in globals() else pl.choices.count("recheck"))
except Exception as e:
    print(e)
print("hybrid (Fig. 3 tracer, corner recovery) done:", HYB_FIG3B)

In [ ]:
# --- hybrid re-run: Fig. 3 tracer, every trace starts with the probe across the edge, recovery 5 x 45 deg, re-check every 4 ---
from selfdriving_afm.planners import gp_edge_explorer_planner
for _m in (gp_edge_explorer_planner, modes):
    importlib.reload(_m)
modes.GPEdgeExplorerPlanner.RECHECK_EVERY = 4
_tv_it0 = cfg.live_view.tv_iterations
cfg.live_view.tv_iterations = 300          # live redraws (~0.2 s, was ~1 s); the figures recompute TV in full
_orig_apply = priors.apply
def _apply_fig3c(cfg_, P_):
    notes = _orig_apply(cfg_, P_)
    for k, v in FOLLOW_FIG3.items():
        setattr(cfg_.edge_explorer, k, v)
    cfg_.edge_explorer.max_features = 5
    cfg_.edge_explorer.follow_recovery_tries = 5
    cfg_.edge_explorer.follow_recovery_turn_deg = 45.0
    return notes
priors.apply = _apply_fig3c
try:
    HYB_FIG3C = edge_run2("gp_edge_explorer", 0.0, "scan_hybrid5fig3c_lam000_gpL3_edgeL2", quiet=3.0)
finally:
    priors.apply = _orig_apply
    cfg.live_view.tv_iterations = _tv_it0
cfg.edge_explorer = _copy.deepcopy(EE0)
for f in planner.features:
    print("feature", f.id, "closed", f.closed_loop, getattr(f, "closed_by", ""), "lost", f.edge_lost, "legs", [len(l) for l in f.legs], "merged", f.merged_ids)
print("hybrid (probe at every start) done:", HYB_FIG3C)

In [ ]:
# --- hybrid: the good run's parameters (Fig. 3 tracer, defaults) + resume a lost trace until the object closes ---
from selfdriving_afm.planners import gp_edge_explorer_planner
for _m in (gp_edge_explorer_planner, modes):
    importlib.reload(_m)
print("RECHECK_EVERY", modes.GPEdgeExplorerPlanner.RECHECK_EVERY, "MAX_RESUMES", modes.GPEdgeExplorerPlanner.MAX_RESUMES)
_tv_it0 = cfg.live_view.tv_iterations
cfg.live_view.tv_iterations = 300
_orig_apply = priors.apply
def _apply_fig3r(cfg_, P_):
    notes = _orig_apply(cfg_, P_)
    for k, v in FOLLOW_FIG3.items():
        setattr(cfg_.edge_explorer, k, v)
    cfg_.edge_explorer.max_features = 5
    return notes
priors.apply = _apply_fig3r
try:
    HYB_R = edge_run2("gp_edge_explorer", 0.0, "scan_hybrid5resume_lam000_gpL3_edgeL2", quiet=3.0)
finally:
    priors.apply = _orig_apply
    cfg.live_view.tv_iterations = _tv_it0
cfg.edge_explorer = _copy.deepcopy(EE0)
for f in planner.features:
    print("feature", f.id, "closed", f.closed_loop, getattr(f, "closed_by", ""), "lost", f.edge_lost, "legs", [len(l) for l in f.legs], "merged", f.merged_ids)
print("resumes:", getattr(planner, "resumes", 0), "| hybrid (resume) done:", HYB_R)

In [ ]:
# --- hybrid: good run's parameters + resume until closed + FIXED edge threshold 58 nm (half the object height) ---
_tv_it0 = cfg.live_view.tv_iterations
cfg.live_view.tv_iterations = 300
_orig_apply = priors.apply
def _apply_fig3t(cfg_, P_):
    notes = _orig_apply(cfg_, P_)
    for k, v in FOLLOW_FIG3.items():
        setattr(cfg_.edge_explorer, k, v)
    cfg_.edge_explorer.max_features = 5
    cfg_.edge_explorer.edge_detect.mode = "manual"
    cfg_.edge_explorer.edge_detect.manual_threshold = 58.0
    return notes
priors.apply = _apply_fig3t
try:
    HYB_T = edge_run2("gp_edge_explorer", 0.0, "scan_hybrid5thr58_lam000_gpL3_edgeL2", quiet=3.0)
finally:
    priors.apply = _orig_apply
    cfg.live_view.tv_iterations = _tv_it0
cfg.edge_explorer = _copy.deepcopy(EE0)
print("threshold used:", planner.substrate.threshold, planner.cfg.edge_detect.mode)
for f in planner.features:
    print("feature", f.id, "closed", f.closed_loop, getattr(f, "closed_by", ""), "lost", f.edge_lost, "legs", [len(l) for l in f.legs], "merged", f.merged_ids)
print("resumes:", getattr(planner, "resumes", 0), "| hybrid (fixed 58 nm) done:", HYB_T)

In [ ]:
# --- LAST hybrid try (fixed 58 nm, resume past the open end) -> then the gamma = 2 edge-focused GP run, automatically ---
import traceback
from selfdriving_afm.planners import gp_path_planner, gp_edge_explorer_planner
for _m in (gp_path_planner, gp_edge_explorer_planner, modes):
    importlib.reload(_m)
print("RECHECK_EVERY", modes.GPEdgeExplorerPlanner.RECHECK_EVERY, "MAX_RESUMES", modes.GPEdgeExplorerPlanner.MAX_RESUMES)
FINAL = {}
_tv_it0 = cfg.live_view.tv_iterations
_orig_apply = priors.apply
def _apply_final(cfg_, P_):
    notes = _orig_apply(cfg_, P_)
    for k, v in FOLLOW_FIG3.items():
        setattr(cfg_.edge_explorer, k, v)
    cfg_.edge_explorer.max_features = 5
    cfg_.edge_explorer.edge_detect.mode = "manual"
    cfg_.edge_explorer.edge_detect.manual_threshold = 58.0
    return notes
try:
    cfg.live_view.tv_iterations = 300
    priors.apply = _apply_final
    FINAL["hybrid"] = edge_run2("gp_edge_explorer", 0.0, "scan_hybridFINAL_thr58_resume_lam000", quiet=3.0)
    for f in planner.features:
        print("feature", f.id, "closed", f.closed_loop, getattr(f, "closed_by", ""), "lost", f.edge_lost, "legs", [len(l) for l in f.legs], "merged", f.merged_ids)
    print("resumes:", getattr(planner, "resumes", 0))
    try:
        import json as _j
        (OUTPUT_DIR / "hybridFINAL_live_log.txt").write_text("\n".join(getattr(live, "log", []) or []))
    except Exception:
        pass
except BaseException as e:
    print("HYBRID FAILED:", repr(e)); traceback.print_exc()
finally:
    priors.apply = _orig_apply
    cfg.live_view.tv_iterations = _tv_it0
    cfg.edge_explorer = _copy.deepcopy(EE0)
    try:
        afm.finish()
    except Exception as e:
        print("finish:", e)
# ---- gamma = 2 ----
_apply_preset0 = apply_gp_preset
def _preset_gamma(cfg_, name):
    out = _apply_preset0(cfg_, name)
    cfg_.gp_path.ivr_interest_power = 2.0
    return out
try:
    apply_gp_preset = _preset_gamma
    FINAL["gamma2"] = gp_run(1.0, tag_prefix="scan_edgefocus_gamma2")
except BaseException as e:
    print("GAMMA2 FAILED:", repr(e)); traceback.print_exc()
finally:
    apply_gp_preset = _apply_preset0
    cfg.gp_path.ivr_interest_power = 1.0
    try:
        afm.finish()
    except Exception as e:
        print("finish:", e)
print("ALL DONE:", FINAL)

In [ ]:
# --- 2026-11-10: two more combinations of levels, Fig. 3 tracer settings, fixed 58 nm threshold ---
#   grid_tracker: level-0 sparse spiral (grid_refine) -> edge tracker on every outline it crossed
#   gp_spiral   : GP (IVR, lambda 0) between objects -> level-1 + level-2 spirals (grid_refine) on each
import importlib
from selfdriving_afm.planners import gp_edge_explorer_planner, hybrid_planners
for _m in (gp_edge_explorer_planner, hybrid_planners, modes):
    importlib.reload(_m)
modes.GPSpiralPlanner.FEATURE_MAX_V = float(PRIORS.feature_max_V)
def hybrid_run(modality, tag_prefix):
    _orig_apply = priors.apply
    def _apply(cfg_, P_):
        notes = _orig_apply(cfg_, P_)
        for k, v in FOLLOW_FIG3.items():
            setattr(cfg_.edge_explorer, k, v)
        ee = cfg_.edge_explorer
        ee.max_features = 5
        ee.edge_detect.mode = "manual"; ee.edge_detect.manual_threshold = 58.0
        if modality == "gp_spiral":           # GP exploring steps as the GP + tracker hybrid
            if ee.follow_segment_length_V is None: ee.follow_segment_length_V = 0.5 * ee.segment_length_V
            if ee.explore_scale_V is None: ee.explore_scale_V = ee.segment_length_V
            ee.segment_length_V = 1.0
        return notes
    _tv0 = cfg.live_view.tv_iterations
    cfg.live_view.tv_iterations = 300
    priors.apply = _apply
    try:
        return edge_run2(modality, 0.0, tag_prefix, quiet=3.0)
    finally:
        priors.apply = _orig_apply
        cfg.live_view.tv_iterations = _tv0
        cfg.edge_explorer = _copy.deepcopy(EE0)
        for f in planner.features:
            print("feature", f.id, "closed", f.closed_loop, f.closed_by, "crossings", len(f.points))
HYB_GRID = hybrid_run("grid_tracker", "scan_hybrid_gridL0_tracker")
print("grid_tracker done:", HYB_GRID)

In [ ]:
# --- GP (lambda 0) between objects -> level-1 + level-2 spirals on each object ---
HYB_GPSPIRAL = hybrid_run("gp_spiral", "scan_hybrid_gp_L2spiral")
print("gp_spiral done:", HYB_GPSPIRAL)

In [ ]:
# --- re-run after finding the cause: the drive-offset correction was SATURATED at correction_max_V = 1.0 V
#     (piezo drift over the day: measured offset now ~(0.45, 1.05) V, up to 1.8 V) -> every spiral / weave
#     landed up to ~1 um off in Y. Cap raised; the run starts from the offset measured last.
import numpy as np
_L = np.array(executor.correction_log)
cfg.position_sensors.correction_max_V = 2.5
cfg.position_sensors.initial_offset_V = tuple(float(v) for v in np.median(_L[-30:, 3:5], axis=0))
print("initial offset", cfg.position_sensors.initial_offset_V, "cap", cfg.position_sensors.correction_max_V)
HYB_GRID2 = hybrid_run("grid_tracker", "scan_hybrid_gridL0_tracker_v2")
print("grid_tracker v2 done:", HYB_GRID2, "| u_target", executor._u_target)

In [ ]:
# --- GP + level-1/2 spirals, re-run with the drive correction un-saturated (cap 2.5 V, start from the last offset) ---
_L = np.array(executor.correction_log)
cfg.position_sensors.initial_offset_V = tuple(float(v) for v in np.median(_L[-30:, 3:5], axis=0))
print("initial offset", cfg.position_sensors.initial_offset_V, "cap", cfg.position_sensors.correction_max_V)
HYB_GPSPIRAL2 = hybrid_run("gp_spiral", "scan_hybrid_gp_L2spiral_v2")
print("gp_spiral v2 done:", HYB_GPSPIRAL2, "| u_target", executor._u_target)

In [ ]:
# --- GP + spirals v3: high trigger (1.55 x threshold, ~90 nm), BIG level-1 spiral (half fmax + 0.8 V),
#     level-2 re-centred and slightly bigger than the object (+0.4 V), image-based closure, disc extended if open ---
_L = np.array(executor.correction_log)
cfg.position_sensors.initial_offset_V = (1.0725, 0.2935)   # last measured (the failed start reset the log)
print("initial offset", cfg.position_sensors.initial_offset_V, "cap", cfg.position_sensors.correction_max_V)
HYB_GPSPIRAL3 = hybrid_run("gp_spiral", "scan_hybrid_gp_L2spiral_v3")
print("gp_spiral v3 done:", HYB_GPSPIRAL3, "| u_target", executor._u_target)

In [ ]:
# --- GP + spirals v4: GP steps 0.5 V, edge probe -> level-1 centred one expected radius inside, level 2 from the
#     level-1 object, size/height checks, quiet -> unexplored holes; decisions also in scan_data/gp_spiral_v4_log.txt ---
_L = np.array(executor.correction_log)
cfg.position_sensors.initial_offset_V = tuple(float(v) for v in np.median(_L[-30:, 3:5], axis=0))
print("initial offset", cfg.position_sensors.initial_offset_V, "cap", cfg.position_sensors.correction_max_V)
HYB_GPSPIRAL4 = hybrid_run("gp_spiral", "scan_hybrid_gp_L2spiral_v4")
print("gp_spiral v4 done:", HYB_GPSPIRAL4, "| u_target", executor._u_target)

In [ ]:
# --- GP + spirals v5: robust degree-2 background (objects trimmed out) refitted before every exploring path,
#     along-path step check + edge-probe step (bow != edge), probe plane gradient -> level-1 centre, local plane
#     inside every spiral, size/height checks, holes. Log: scan_data/gp_spiral_v5_log.txt ---
_L = np.array(executor.correction_log)
cfg.position_sensors.initial_offset_V = tuple(float(v) for v in np.median(_L[-30:, 3:5], axis=0))
print("initial offset", cfg.position_sensors.initial_offset_V, "cap", cfg.position_sensors.correction_max_V)
HYB_GPSPIRAL5 = hybrid_run("gp_spiral", "scan_hybrid_gp_L2spiral_v5")
print("gp_spiral v5 done:", HYB_GPSPIRAL5, "| u_target", executor._u_target)

In [ ]:
# --- GP + spirals v6: v5 + edge probe re-centred on where it actually landed (position-dependent drive error),
#     probe radius 0.5 V, level-1 object isolated by mean-shift (no merging of neighbours). Log: scan_data/gp_spiral_v6_log.txt ---
_L = np.array(executor.correction_log)
cfg.position_sensors.initial_offset_V = tuple(float(v) for v in np.median(_L[-30:, 3:5], axis=0))
print("initial offset", cfg.position_sensors.initial_offset_V, "cap", cfg.position_sensors.correction_max_V)
HYB_GPSPIRAL6 = hybrid_run("gp_spiral", "scan_hybrid_gp_L2spiral_v6")
print("gp_spiral v6 done:", HYB_GPSPIRAL6, "| u_target", executor._u_target)

In [ ]:
# --- the equivalent raster for gp_spiral v6 (skipped after the run for RAM; memory freed first) ---
with afm.run("equivalent raster (user-defined)", executor):
    REF_ = user_raster.acquire(RP6, executor, cfg)
reference_raster.save_into(HYB_GPSPIRAL6, REF_); del REF_
gc.collect()
print("raster saved into", HYB_GPSPIRAL6)